# KA-STGAT Colab training (AMP + GPU)

1. Runtime → GPU.
2. Upload this repo or clone it.
3. Run all. Copy `checkpoints/last.ckpt` back to the laptop / Drive.

In [ ]:
import os, sys, subprocess, pathlib
root = pathlib.Path("/content/Engage2")
if not (root / "pyproject.toml").exists():
    # If you uploaded a zip, extract it; otherwise expect the repo already mounted.
    print("Place the Engage2 repo at /content/Engage2 or change ROOT below.")
ROOT = next((p for p in [root, pathlib.Path.cwd()] if (p / "pyproject.toml").exists()), pathlib.Path.cwd())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("ROOT", ROOT.resolve())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightning", "torch-geometric", "pyyaml", "pyarrow", "scipy", "scikit-learn"])

In [ ]:
import torch, lightning as L
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "Switch the runtime to GPU"

In [ ]:
from kastgat.data.clean import clean_tracks
from kastgat.data.datamodule import make_loaders, save_samples
from kastgat.data.graphs import build_windows
from kastgat.data.sources.synthetic import generate_synthetic_traffic, SectorSpec
from kastgat.data.sources.bluesky_sim import generate_labeled_from_scn_commands
from kastgat.utils.config import load_config

cfg = load_config()
syn = clean_tracks(generate_synthetic_traffic(SectorSpec(n_aircraft=12, duration_s=900, seed=0)))
blu = clean_tracks(generate_labeled_from_scn_commands(duration_s=600))
import pandas as pd
df = pd.concat([syn, blu], ignore_index=True)
samples = build_windows(df, history_steps=cfg["history_steps"], horizon_steps=cfg["horizon_steps"],
                        max_nodes=cfg["max_nodes"], proximity_nm=cfg["proximity_nm"])
print("windows", len(samples), "rows", len(df))
save_samples(samples, ROOT / "data" / "processed" / "windows.pt")
df.to_parquet(ROOT / "data" / "processed" / "canonical.parquet", index=False)
train_loader, val_loader = make_loaders(samples, batch_size=8)

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from kastgat.train.lit_module import KASTGATModule

L.seed_everything(42)
module = KASTGATModule(cfg)
ckpt_dir = ROOT / "checkpoints"
ckpt_dir.mkdir(exist_ok=True)
trainer = L.Trainer(
    max_epochs=20,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    callbacks=[
        ModelCheckpoint(dirpath=ckpt_dir, monitor="val/loss", mode="min", save_last=True),
        EarlyStopping(monitor="val/loss", patience=5, mode="min"),
    ],
    log_every_n_steps=5,
)
trainer.fit(module, train_loader, val_loader)
trainer.save_checkpoint(ckpt_dir / "last.ckpt")
print("wrote", ckpt_dir / "last.ckpt")

Download `checkpoints/last.ckpt` to the laptop. The Streamlit app loads it automatically.